In [1]:
# ============================================================================
# Drought characterization — Sao Francisco basin
# Method: Run Theory (Yevjevich, 1967)
# Index: SPEI-6
# Metrics: Frequency, Duration (median), Severity (median)
# Definition: event starts when SPEI < -1 for >= 2 consecutive months
#             event ends when SPEI >= 0
# Reference: Spinoni et al. (2014), Tabari et al. (2021)
# Scenarios: Historical (1985-2014), SSP126 (2041-2070), SSP585 (2041-2070)
# Models: MPI-ESM1-2-HR, UKESM1-0-LL, GFDL-ESM4, IPSL-CM6A-LR, MRI-ESM2-0
# ============================================================================

import os
import numpy as np
import xarray as xr
import netCDF4 as nc
from pathlib import Path

# --- CONFIGURATION ---
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "sao_francisco"

path_in  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/3.outputs_SPEI_R/5.baseline_106_6ac_OF/Sao_Francisco/"
path_out = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/"


# ============================================================================
# FUNCTION: Calculate drought metrics for a single time series
# ============================================================================
def calc_drought_metrics(spei_ts):
    """
    Calculates drought frequency, duration and severity using run theory.

    Parameters:
        spei_ts (np.array): SPEI-6 time series for a single pixel

    Returns:
        frequency (int):         number of drought events
        duration_median (float): median duration across all events (months)
        severity_median (float): median severity across all events (sum of |SPEI|)

    Event definition:
        Start: SPEI < -1 for >= 2 consecutive months
        End:   SPEI >= 0
    """
    n           = len(spei_ts)
    in_event    = False
    consec_neg  = 0
    event_start = None
    durations   = []
    severities  = []

    for t in range(n):
        val = spei_ts[t]

        if np.isnan(val):
            in_event   = False
            consec_neg = 0
            continue

        if not in_event:
            if val < -1:
                consec_neg += 1
                if consec_neg >= 2:
                    in_event    = True
                    event_start = t - 1
            else:
                consec_neg = 0
        else:
            if val >= 0:
                duration = (t - 1) - event_start + 1
                severity = np.sum(np.abs(spei_ts[event_start:t]))
                durations.append(duration)
                severities.append(severity)
                in_event   = False
                consec_neg = 0

    # Handle event that does not close before end of series
    if in_event:
        duration = (n - 1) - event_start + 1
        severity = np.sum(np.abs(spei_ts[event_start:n]))
        durations.append(duration)
        severities.append(severity)

    frequency = len(durations)

    if frequency == 0:
        return frequency, np.nan, np.nan

    return frequency, np.median(durations), np.median(severities)


# ============================================================================
# FUNCTION: Save drought metrics as NetCDF
# ============================================================================
def save_netcdf(freq_grid, dur_grid, sev_grid, lon, lat,
                basin, scenario, model, output_dir):
    """
    Saves drought metrics (frequency, duration, severity) into a single NetCDF.
    Arrays are stored in (lat, lon) order following CF conventions.
    """
    output_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
    ds_out      = nc.Dataset(output_file, "w", format="NETCDF4")

    ds_out.createDimension("lat", len(lat))
    ds_out.createDimension("lon", len(lon))

    lat_var       = ds_out.createVariable("lat", "f4", ("lat",))
    lat_var.units = "degrees_north"
    lat_var[:]    = lat

    lon_var       = ds_out.createVariable("lon", "f4", ("lon",))
    lon_var.units = "degrees_east"
    lon_var[:]    = lon

    freq_save = np.where(np.isnan(freq_grid), -9999, freq_grid)
    dur_save  = np.where(np.isnan(dur_grid),  -9999, dur_grid)
    sev_save  = np.where(np.isnan(sev_grid),  -9999, sev_grid)

    freq_var           = ds_out.createVariable("frequency", "f4", ("lat", "lon"),
                                                fill_value=-9999)
    freq_var.units     = "count"
    freq_var.long_name = "Number of drought events"
    freq_var[:]        = freq_save

    dur_var           = ds_out.createVariable("duration", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    dur_var.units     = "months"
    dur_var.long_name = "Median drought duration"
    dur_var[:]        = dur_save

    sev_var           = ds_out.createVariable("severity", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    sev_var.units     = "-"
    sev_var.long_name = "Median drought severity (sum of |SPEI| per event)"
    sev_var[:]        = sev_save

    ds_out.description = (f"Drought characteristics — {basin} basin, "
                          f"{scenario}, {model}. "
                          f"Run theory (Yevjevich 1967): "
                          f"start SPEI < -1 for >= 2 months, end SPEI >= 0.")
    ds_out.index       = "SPEI-6"
    ds_out.basin       = basin
    ds_out.scenario    = scenario
    ds_out.model       = model

    ds_out.close()
    print(f"    Saved: {output_file}")


# ============================================================================
# FUNCTION: Calculate ensemble median across 5 models
# ============================================================================
def calc_ensemble(basin, scenario, models, output_dir):
    """
    Reads individual model NetCDFs and calculates pixel-wise median
    across all models for each metric.
    """
    freq_stack = []
    dur_stack  = []
    sev_stack  = []

    for model in models:
        nc_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)

        freq = ds["frequency"].values.astype(float)
        dur  = ds["duration"].values.astype(float)
        sev  = ds["severity"].values.astype(float)

        freq = np.where(freq == -9999, np.nan, freq)
        dur  = np.where(dur  == -9999, np.nan, dur)
        sev  = np.where(sev  == -9999, np.nan, sev)

        freq_stack.append(freq)
        dur_stack.append(dur)
        sev_stack.append(sev)

        lon = ds["lon"].values
        lat = ds["lat"].values
        ds.close()

    freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
    dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
    sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)

    save_netcdf(freq_ensemble, dur_ensemble, sev_ensemble,
                lon, lat, basin, scenario, "ensemble", output_dir)

    print(f"\n  Ensemble summary — {scenario}:")
    for name, grid in [("FREQUENCY", freq_ensemble),
                       ("DURATION",  dur_ensemble),
                       ("SEVERITY",  sev_ensemble)]:
        valid = grid[~np.isnan(grid)]
        print(f"    {name}: min={valid.min():.2f}, "
              f"max={valid.max():.2f}, "
              f"median={np.median(valid):.2f}")


# ============================================================================
# MAIN LOOP — SCENARIOS x MODELS
# ============================================================================
for scenario in scenarios:

    out_dir = f"{path_out}{scenario}/"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario.upper()} | BASIN: {basin.upper()}")
    print(f"{'='*60}")

    for model in models:

        print(f"\n  Processing: {model}")

        # Read SPEI-6 NetCDF
        nc_file = f"{path_in}{scenario}/spei6_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)
        spei    = ds["SPEI_6"].values
        lon     = ds["lon"].values
        lat     = ds["lat"].values
        dims    = ds["SPEI_6"].dims
        ds.close()

        spei = np.where(spei == -9999, np.nan, spei)

        if dims[0] == "time":
            spei = np.transpose(spei, (1, 2, 0))

        n_lon, n_lat, _ = spei.shape

        freq_grid = np.full((n_lon, n_lat), np.nan)
        dur_grid  = np.full((n_lon, n_lat), np.nan)
        sev_grid  = np.full((n_lon, n_lat), np.nan)

        for i in range(n_lon):
            for j in range(n_lat):
                ts = spei[i, j, :]
                if np.all(np.isnan(ts)):
                    continue
                (freq_grid[i, j],
                 dur_grid[i, j],
                 sev_grid[i, j]) = calc_drought_metrics(ts)

        for name, grid in [("FREQUENCY", freq_grid),
                           ("DURATION",  dur_grid),
                           ("SEVERITY",  sev_grid)]:
            valid = grid[~np.isnan(grid)]
            print(f"    {name}: min={valid.min():.2f}, "
                  f"max={valid.max():.2f}, "
                  f"median={np.median(valid):.2f}")

        save_netcdf(freq_grid, dur_grid, sev_grid,
                    lon, lat, basin, scenario, model, out_dir)

    print(f"\n  Computing ensemble median — {scenario}")
    calc_ensemble(basin, scenario, models, out_dir)


print(f"\n{'='*60}")
print(f"COMPLETED — Sao Francisco basin, 5 models, 3 scenarios")
print(f"Output: {path_out}")
print(f"{'='*60}")


SCENARIO: HISTORICAL | BASIN: SAO_FRANCISCO

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=4.00, max=16.00, median=9.00
    DURATION: min=5.50, max=29.00, median=10.00
    SEVERITY: min=4.85, max=35.66, median=10.89
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/historical/drought_metrics_sao_francisco_historical_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=6.00, max=16.00, median=10.00
    DURATION: min=5.50, max=25.50, median=9.00
    SEVERITY: min=5.77, max=27.00, median=10.55
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/historical/drought_metrics_sao_francisco_historical_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=5.00, max=14.00, median=9.00
    DURATION: min=6.00, max=26.00, median=11.50
    SEVERITY: min=5.44, max=30.21, median=12.40
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequi

/tmp/ipykernel_1960958/2554593537.py:184: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:185: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:186: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/historical/drought_metrics_sao_francisco_historical_ensemble.nc

  Ensemble summary — historical:
    FREQUENCY: min=7.00, max=13.00, median=9.00
    DURATION: min=7.00, max=20.00, median=10.50
    SEVERITY: min=7.25, max=20.04, median=11.26

SCENARIO: SSP126 | BASIN: SAO_FRANCISCO

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=6.00, max=16.00, median=11.00
    DURATION: min=5.00, max=24.00, median=10.00
    SEVERITY: min=5.31, max=29.22, median=12.10
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/ssp126/drought_metrics_sao_francisco_ssp126_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=5.00, max=16.00, median=10.00
    DURATION: min=6.00, max=43.00, median=13.00
    SEVERITY: min=6.87, max=52.57, median=17.06
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SP

/tmp/ipykernel_1960958/2554593537.py:184: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:185: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:186: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    FREQUENCY: min=6.00, max=16.00, median=10.00
    DURATION: min=6.00, max=42.00, median=14.00
    SEVERITY: min=6.82, max=46.46, median=15.88
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/ssp585/drought_metrics_sao_francisco_ssp585_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=3.00, max=14.00, median=8.00
    DURATION: min=6.50, max=92.00, median=18.00
    SEVERITY: min=6.94, max=109.07, median=23.17
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/ssp585/drought_metrics_sao_francisco_ssp585_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=6.00, max=17.00, median=11.00
    DURATION: min=7.00, max=27.50, median=15.50
    SEVERITY: min=9.00, max=36.78, median=20.23
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/ssp585/drought_metrics_sao_francisco_ssp585_GFDL-ESM4.

/tmp/ipykernel_1960958/2554593537.py:184: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:185: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1960958/2554593537.py:186: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


In [2]:
# ============================================================================
# Verification — Sao Francisco basin — SPEI-6
# ============================================================================

import numpy as np
import xarray as xr
from pathlib import Path

path_out  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Sao_Francisco/"
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0", "ensemble"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "sao_francisco"

print("=" * 60)
print("NETCDF VERIFICATION — Sao Francisco basin — SPEI-6")
print("=" * 60)

all_ok = True

for scenario in scenarios:
    print(f"\n--- {scenario.upper()} ---")
    for model in models:
        nc_file = Path(f"{path_out}{scenario}/drought_metrics_{basin}_{scenario}_{model}.nc")

        if not nc_file.exists():
            print(f"  ❌ MISSING: {nc_file.name}")
            all_ok = False
            continue

        ds   = xr.open_dataset(nc_file)
        freq = np.where(ds["frequency"].values == -9999, np.nan, ds["frequency"].values.astype(float))
        dur  = np.where(ds["duration"].values  == -9999, np.nan, ds["duration"].values.astype(float))
        sev  = np.where(ds["severity"].values  == -9999, np.nan, ds["severity"].values.astype(float))
        lon  = ds["lon"].values
        lat  = ds["lat"].values
        ds.close()

        n_valid   = np.sum(~np.isnan(freq))
        shape_ok  = freq.ndim == 2
        pixels_ok = (np.sum(~np.isnan(freq)) == np.sum(~np.isnan(dur)) == np.sum(~np.isnan(sev)))
        values_ok = (np.all(freq[~np.isnan(freq)] >= 0) and
                     np.all(dur[~np.isnan(dur)]   >= 0) and
                     np.all(sev[~np.isnan(sev)]   >= 0))
        no_inf    = not (np.any(np.isinf(freq)) or np.any(np.isinf(dur)) or np.any(np.isinf(sev)))

        status = "✅" if (shape_ok and pixels_ok and values_ok and no_inf) else "❌"
        if status == "❌":
            all_ok = False

        print(f"  {status} {model}")
        print(f"       Shape: {freq.shape} | Valid pixels: {n_valid} | "
              f"Freq=[{freq[~np.isnan(freq)].min():.1f},{freq[~np.isnan(freq)].max():.1f}] | "
              f"Dur=[{dur[~np.isnan(dur)].min():.1f},{dur[~np.isnan(dur)].max():.1f}] | "
              f"Sev=[{sev[~np.isnan(sev)].min():.2f},{sev[~np.isnan(sev)].max():.2f}]")

print(f"\n{'='*60}")
print("✅ ALL FILES VERIFIED — SPEI-6" if all_ok else "❌ SOME FILES HAVE ISSUES")
print(f"{'='*60}")

NETCDF VERIFICATION — Sao Francisco basin — SPEI-6

--- HISTORICAL ---
  ✅ MPI-ESM1-2-HR
       Shape: (40, 25) | Valid pixels: 556 | Freq=[4.0,16.0] | Dur=[5.5,29.0] | Sev=[4.85,35.66]
  ✅ UKESM1-0-LL
       Shape: (40, 25) | Valid pixels: 556 | Freq=[6.0,16.0] | Dur=[5.5,25.5] | Sev=[5.77,27.00]
  ✅ GFDL-ESM4
       Shape: (40, 25) | Valid pixels: 556 | Freq=[5.0,14.0] | Dur=[6.0,26.0] | Sev=[5.44,30.21]
  ✅ IPSL-CM6A-LR
       Shape: (40, 25) | Valid pixels: 556 | Freq=[5.0,16.0] | Dur=[5.0,24.0] | Sev=[4.73,28.84]
  ✅ MRI-ESM2-0
       Shape: (40, 25) | Valid pixels: 556 | Freq=[5.0,12.0] | Dur=[7.0,26.0] | Sev=[6.18,33.36]
  ✅ ensemble
       Shape: (40, 25) | Valid pixels: 556 | Freq=[7.0,13.0] | Dur=[7.0,20.0] | Sev=[7.25,20.04]

--- SSP126 ---
  ✅ MPI-ESM1-2-HR
       Shape: (40, 25) | Valid pixels: 556 | Freq=[6.0,16.0] | Dur=[5.0,24.0] | Sev=[5.31,29.22]
  ✅ UKESM1-0-LL
       Shape: (40, 25) | Valid pixels: 556 | Freq=[5.0,16.0] | Dur=[6.0,43.0] | Sev=[6.87,52.57]
  ✅ GFDL-E